# Chapter 12 — Exercises: Worked Solutions

Worked solutions for Chapter 12 (Advanced Topics and Future Directions).

**Exercises:**
1. CPA vs PC-SAFT for methanol VLE
2. Electrolyte CPA concept: NaCl activity
3. Model selection decision framework

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim


NeqSim mode: pip


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314

SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

---
## Exercise 12.1 — CPA vs SRK for Methanol–Hexane

**Problem:** Compute and compare the VLE of methanol–n-hexane at 1 atm
using (a) SRK (no association) and (b) CPA. This is a classic test
because the system shows a minimum boiling azeotrope.

### Solution

In [3]:
x_meoh = np.arange(0.05, 0.96, 0.05)
T_bp_cpa, T_bp_srk = [], []

for xm in x_meoh:
    # CPA
    try:
        f = SystemSrkCPAstatoil(273.15 + 60.0, 1.01325)
        f.addComponent("methanol", float(xm))
        f.addComponent("n-hexane", float(1.0 - xm))
        f.setMixingRule(10)
        ops = ThermodynamicOperations(f)
        ops.bubblePointTemperatureFlash()
        T_bp_cpa.append(float(f.getTemperature()) - 273.15)
    except Exception:
        T_bp_cpa.append(np.nan)
    # SRK
    try:
        f2 = SystemSrkEos(273.15 + 60.0, 1.01325)
        f2.addComponent("methanol", float(xm))
        f2.addComponent("n-hexane", float(1.0 - xm))
        f2.setMixingRule("classic")
        ops2 = ThermodynamicOperations(f2)
        ops2.bubblePointTemperatureFlash()
        T_bp_srk.append(float(f2.getTemperature()) - 273.15)
    except Exception:
        T_bp_srk.append(np.nan)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(x_meoh[:len(T_bp_cpa)], T_bp_cpa, 'o-', color=BLUE, markersize=3, linewidth=1.2, label="CPA")
ax.plot(x_meoh[:len(T_bp_srk)], T_bp_srk, 's--', color=ORANGE, markersize=3, linewidth=1.0, label="SRK")
ax.set_xlabel("$x_{MeOH}$"); ax.set_ylabel("Bubble point T (\u00b0C)")
ax.set_title("Exercise 12.1: MeOH\u2013hexane Txy")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch12_ex01_meoh_hexane.png")

Saved: fig_ch12_ex01_meoh_hexane.png


**Answer:** CPA correctly predicts the minimum boiling azeotrope at
~50\u00b0C because it accounts for methanol's self-association. SRK
without association fails to capture the strong positive deviation from
Raoult's law that causes the azeotrope.

---
## Exercise 12.2 — Electrolyte Effects: Water Activity with NaCl

**Problem:** Explain conceptually how the Debye\u2013H\u00fcckel term in e-CPA
affects water activity. Compute water activity depression using
Raoult's law analogy for different salt concentrations.

### Solution

The Debye\u2013H\u00fcckel contribution lowers the chemical potential of ions in
solution, which reduces water activity through the Gibbs\u2013Duhem relation.
At molality $m$, the Debye\u2013H\u00fcckel limiting law gives:

$$\ln \gamma_{\pm} = -A \sqrt{I}$$

where $A \approx 1.172$ (mol/kg)$^{-1/2}$ at 25\u00b0C and $I$ is ionic strength.

In [4]:
# Debye-Huckel model for water activity
A_DH = 1.172  # (mol/kg)^-1/2 at 25 C
m_nacl = np.linspace(0.01, 6.0, 100)  # molality
nu = 2  # NaCl dissociates into 2 ions
M_w = 0.018015  # kg/mol

# Ionic strength
I = m_nacl  # for 1-1 electrolyte

# Osmotic coefficient (Debye-Huckel)
phi_os = 1 - (2 * A_DH / 3) * np.sqrt(I)

# Water activity from osmotic coefficient
a_w = np.exp(-nu * m_nacl * M_w * phi_os)

# Ideal (Raoult's law)
x_w = 1.0 / (1.0 + nu * m_nacl * M_w)

# Experimental data (Robinson & Stokes)
m_exp = [0.5, 1.0, 2.0, 3.0, 4.0, 5.0]
aw_exp = [0.983, 0.967, 0.932, 0.896, 0.858, 0.819]

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(m_nacl, a_w, color=BLUE, linewidth=1.5, label="DH limiting law")
ax.plot(m_nacl, x_w, color=GREEN, linewidth=1.0, linestyle="--", label="Raoult's law")
ax.scatter(m_exp, aw_exp, color="black", s=25, zorder=5, label="Exp.")
ax.set_xlabel("NaCl molality (mol/kg)"); ax.set_ylabel("Water activity $a_w$")
ax.set_title("Exercise 12.2: Salt effect on $a_w$")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
ax.set_ylim(0.7, 1.01)
fig.tight_layout()
save(fig, "fig_ch12_ex02_nacl_activity.png")

Saved: fig_ch12_ex02_nacl_activity.png


**Answer:** The DH limiting law captures the initial decrease in water
activity but deviates at higher concentrations. The full e-CPA model
adds Born solvation and ion-specific association terms to extend
accuracy up to 6+ mol/kg. Water activity depression is critical for
hydrate inhibition: lower $a_w$ shifts hydrate curves to lower temperatures.

---
## Exercise 12.3 — Model Selection Framework

**Problem:** Given three industrial scenarios, recommend the appropriate
thermodynamic model (SRK, CPA, e-CPA, or PC-SAFT) and justify:

1. Natural gas dehydration with TEG
2. CO\u2082 injection in a saline aquifer
3. Lean hydrocarbon gas compression

### Solution

In [5]:
# Decision matrix visualization
scenarios = ["TEG dehydration", "CO\u2082+brine (CCS)", "Lean gas compression"]
models = ["SRK", "CPA", "e-CPA", "PC-SAFT"]

# Score: 0 = not suitable, 1 = adequate, 2 = recommended
scores = np.array([
    [0, 2, 2, 1],  # TEG dehydration
    [0, 1, 2, 1],  # CO2 + brine
    [2, 2, 2, 2],  # Lean gas
])

fig, ax = plt.subplots(figsize=(3.5, 2.4))
cmap = plt.cm.RdYlGn
im = ax.imshow(scores, cmap=cmap, vmin=0, vmax=2, aspect="auto")
ax.set_xticks(range(len(models))); ax.set_xticklabels(models, fontsize=8)
ax.set_yticks(range(len(scenarios))); ax.set_yticklabels(scenarios, fontsize=8)
for i in range(len(scenarios)):
    for j in range(len(models)):
        labels = ["\u2717", "\u2713", "\u2713\u2713"]
        ax.text(j, i, labels[scores[i, j]], ha="center", va="center", fontsize=10,
                color="white" if scores[i, j] == 0 else "black")
ax.set_title("Exercise 12.3: Model selection", fontsize=9)
fig.tight_layout()
save(fig, "fig_ch12_ex03_model_selection.png")

Saved: fig_ch12_ex03_model_selection.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_39076\1481331562.py:23: UserWarning: Glyph 10007 (\N{BALLOT X}) missing from font(s) DejaVu Serif.
  fig.tight_layout()
C:\Users\ESOL\AppData\Local\Temp\ipykernel_39076\1481331562.py:23: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) DejaVu Serif.
  fig.tight_layout()
C:\Users\ESOL\AppData\Local\Temp\ipykernel_39076\99709758.py:10: UserWarning: Glyph 10007 (\N{BALLOT X}) missing from font(s) DejaVu Serif.
  def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
C:\Users\ESOL\AppData\Local\Temp\ipykernel_39076\99709758.py:10: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) DejaVu Serif.
  def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")


**Answer:**

1. **TEG dehydration** \u2192 **CPA** (recommended) or e-CPA. TEG and water
   are both associating compounds; SRK cannot capture the strong
   hydrogen bonding and gives poor water content predictions.

2. **CO\u2082 + saline aquifer** \u2192 **e-CPA** (recommended). The presence
   of dissolved salts (NaCl, CaCl\u2082) requires the electrolyte extension.
   CPA alone misses the salting-out effect on CO\u2082 solubility.

3. **Lean gas compression** \u2192 **SRK** (simplest adequate model). No
   associating compounds; all models give comparable results, but SRK
   is fastest and well-validated for light hydrocarbons.